In [3]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# === 路径设置 ===
base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata")
grid_path = base_dir / "Jiangsu_CGCS2000/Jiangsu_grid_500m_ID.gpkg"
fvc_dir = base_dir / "Jiangsu_fvc"     # ← 这里是你的FVC文件夹
out_path = base_dir / "Jiangsu_CGCS2000/Jiangsu_grid_fvcavg_2012_2022.gpkg"

# === 读取鱼网 ===
grid = gpd.read_file(grid_path)
print(f"✅ Loaded {len(grid)} grids")

# === 循环处理各年份 ===
years = range(2012, 2023)
for y in tqdm(years):
    fvc_file = fvc_dir / f"FVC_Jiangsu_{y}_cropland_only_4547.tif"
    if not fvc_file.exists():
        print(f"⚠️ Missing file: {fvc_file.name}, skipped.")
        continue

    print(f"→ Processing {y}: {fvc_file.name}")
    with rasterio.open(fvc_file) as src:
        stats = zonal_stats(
            grid,
            fvc_file,
            stats=["mean"],
            nodata=0,           # 非耕地为0
            geojson_out=False,
            raster_out=False
        )
        grid[f"fvc_{y}"] = [s["mean"] for s in stats]

# === 保存输出 ===
grid.to_file(out_path, driver="GPKG")
grid.drop(columns="geometry").to_csv(out_path.with_suffix(".csv"), index=False)
print(f"✅ Finished and saved: {out_path}")


✅ Loaded 410620 grids


  0%|          | 0/11 [00:00<?, ?it/s]

→ Processing 2012: FVC_Jiangsu_2012_cropland_only_4547.tif


  9%|▉         | 1/11 [07:53<1:18:52, 473.28s/it]

→ Processing 2013: FVC_Jiangsu_2013_cropland_only_4547.tif


...

→ Processing 2021: FVC_Jiangsu_2021_cropland_only_4547.tif


 91%|█████████ | 10/11 [1:48:04<11:25, 685.45s/it]

→ Processing 2022: FVC_Jiangsu_2022_cropland_only_4547.tif


100%|██████████| 11/11 [1:56:28<00:00, 635.29s/it]


✅ Finished and saved: /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_fvcavg_2012_2022.gpkg


In [8]:
import geopandas as gpd
from pathlib import Path

# === 路径设置 ===
base = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000")
grid_path = base / "Jiangsu_grid_500m_ID.gpkg"
china_path = Path("/Users/wangze/Dropbox/Emi/china.shp")
out_path = base / "Jiangsu_grid_with_admin.gpkg"

# === 读取数据 ===
grid = gpd.read_file(grid_path)
china = gpd.read_file(china_path, encoding="gbk", errors="ignore")

print(f"✅ Grid loaded: {len(grid)} cells, CRS = {grid.crs}")
print(f"✅ China shapefile loaded: {len(china)} polygons, CRS = {china.crs}")

# === 统一坐标系统 ===
# 如果china没有crs，则假定为EPSG:4326
if china.crs is None:
    china.set_crs(epsg=4326, inplace=True)
    print("⚠️ CRS for china was undefined — assumed EPSG:4326")

# 若china为4326，则转为4547
if china.crs.to_epsg() != 4547:
    print(f"🔄 Converting china CRS from {china.crs} → EPSG:4547")
    china = china.to_crs(epsg=4547)

# 若grid不是4547，也统一
if grid.crs.to_epsg() != 4547:
    print(f"🔄 Converting grid CRS from {grid.crs} → EPSG:4547")
    grid = grid.to_crs(epsg=4547)

print(f"✅ CRS unified → {grid.crs}")

# === 空间拼接 ===
print("⏳ Performing spatial join...")
joined = gpd.sjoin(grid, china, how="left", predicate="intersects")

# === 清理字段 ===
joined = joined.drop(columns=[c for c in joined.columns if c.startswith("index_")], errors="ignore")

# === 保存输出 ===
joined.to_file(out_path, driver="GPKG")
joined.drop(columns="geometry").to_csv(out_path.with_suffix(".csv"), index=False, encoding="utf-8-sig")

print(f"✅ Saved with admin info → {out_path}")
print(f"✅ Output records: {len(joined)}")



/Users/wangze/Dropbox/Emi/LandControl/.venv/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver ESRI Shapefile does not support open option ERRORS
  return ogr_read(
/Users/wangze/Dropbox/Emi/LandControl/.venv/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: One or several characters couldn't be converted correctly from gbk to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


✅ Grid loaded: 410620 cells, CRS = EPSG:4547
✅ China shapefile loaded: 2901 polygons, CRS = EPSG:4326
🔄 Converting china CRS from EPSG:4326 → EPSG:4547
✅ CRS unified → EPSG:4547
⏳ Performing spatial join...
✅ Saved with admin info → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_with_admin.gpkg
✅ Output records: 430658
